# WTDF · Run Preprocessing

This notebook runs the preprocessing workflow for the wind turbine fault detection project using the refactored `WindFarmProcessor`.

## Objectives
- Validate project paths and preprocessing configuration
- Process raw SCADA event CSV files into event-level Parquet files
- Build a consolidated `master_dataset.parquet` using the memory-aware PyArrow workflow
- Inspect lightweight metadata and small samples without loading the full master dataset into memory

## Notes
- Core preprocessing logic lives in `src/wtfd/data/preprocessing.py`
- This notebook is intentionally thin and orchestration-focused
- The master dataset creation step now returns a **path**, not a full DataFrame
- To reduce memory pressure, this notebook avoids keeping large DataFrames alive unnecessarily

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys
import gc

import pandas as pd
import pyarrow.parquet as pq

In [3]:
# Ensure the project root is on the Python path when running from /notebooks
PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project')

In [4]:
from wtfd.data.preprocessing import WindFarmProcessor
from wtfd.utils.logging_utils import get_logger

## Define project paths and preprocessing settings

In [5]:
CONFIG_PATH = PROJECT_ROOT / "config" / "feature_map.yaml"
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "zenodo_windfarm_data"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
MASTER_DATASET_PATH = PROCESSED_PATH / "master_dataset.parquet"

LEAD_IN_DAYS = 1.0
CREATE_MASTER_DATASET = True

CONFIG_PATH, RAW_PATH, PROCESSED_PATH, MASTER_DATASET_PATH

(PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/config/feature_map.yaml'),
 PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/raw/zenodo_windfarm_data'),
 PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed'),
 PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/master_dataset.parquet'))

In [6]:
logger = get_logger("wtfd.notebooks.run_preprocessing")
logger.info("Notebook preprocessing session initialized")

2026-03-20 16:51:09 | INFO | wtfd.notebooks.run_preprocessing | Notebook preprocessing session initialized


## Validate required inputs

In [7]:
assert CONFIG_PATH.exists(), f"Missing config file: {CONFIG_PATH}"
assert RAW_PATH.exists(), f"Missing raw data path: {RAW_PATH}"

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Config file:", CONFIG_PATH)
print("Raw path:", RAW_PATH)
print("Processed path:", PROCESSED_PATH)
print("Master dataset path:", MASTER_DATASET_PATH)

Config file: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/config/feature_map.yaml
Raw path: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/raw/zenodo_windfarm_data
Processed path: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed
Master dataset path: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/master_dataset.parquet


## Initialize preprocessing pipeline

In [8]:
processor = WindFarmProcessor(
    config_path=CONFIG_PATH,
    lead_in_days=LEAD_IN_DAYS,
)

processor

2026-03-20 16:51:09 | INFO | wtfd.data.preprocessing | Initialized WindFarmProcessor with config_path=/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/config/feature_map.yaml and lead_in_days=1.0


## Run event-level preprocessing

This step processes all raw turbine event files into standardized event-level Parquet files.

In [9]:
processor.process_all_turbines(
    raw_data_root=RAW_PATH,
    output_dir=PROCESSED_PATH,
)

gc.collect()

2026-03-20 16:51:09 | INFO | wtfd.data.preprocessing | Beginning batch preprocessing from /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/raw/zenodo_windfarm_data to /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed
2026-03-20 16:51:09 | INFO | wtfd.data.preprocessing | Found 22 raw files for farm A
Farm A:   0%|                                                                                        | 0/22 [00:00<?, ?it/s]2026-03-20 16:51:09 | INFO | wtfd.data.preprocessing | Starting pipeline for farm_id=A, file=0.csv
2026-03-20 16:51:11 | INFO | wtfd.data.preprocessing | Completed pipeline for farm_id=A, file=0.csv, output_shape=(54986, 39)
2026-03-20 16:51:12 | INFO | wtfd.data.preprocessing | Saved processed file: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/A_event_0.parquet
Farm A:   5%|███▋                                                                            | 1/22 [00:02<00:58,  2.77s

0

## Review generated event-level Parquet files

In [10]:
event_parquet_files = sorted(
    [
        f
        for f in PROCESSED_PATH.glob("*.parquet")
        if f.name != "master_dataset.parquet"
    ]
)

print("Number of event parquet files:", len(event_parquet_files))
event_parquet_files[:10]

Number of event parquet files: 95


[PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/A_event_0.parquet'),
 PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/A_event_10.parquet'),
 PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/A_event_13.parquet'),
 PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/A_event_14.parquet'),
 PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/A_event_17.parquet'),
 PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/A_event_22.parquet'),
 PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/A_event_24.parquet'),
 PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/A_event_25.parquet'),
 PosixPath('/mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/dat

In [11]:
event_file_summary = pd.DataFrame(
    {
        "file_name": [f.name for f in event_parquet_files],
        "size_mb": [round(f.stat().st_size / (1024 ** 2), 3) for f in event_parquet_files],
    }
)

event_file_summary.head(10)

,file_name,size_mb
0,A_event_0.parquet,7.358
1,A_event_10.parquet,7.203
2,A_event_13.parquet,7.197
3,A_event_14.parquet,7.326
4,A_event_17.parquet,7.411
5,A_event_22.parquet,7.057
6,A_event_24.parquet,7.403
7,A_event_25.parquet,7.369
8,A_event_26.parquet,7.199
9,A_event_3.parquet,7.451


## Build consolidated master dataset

This uses the memory-aware PyArrow dataset workflow and returns the saved output path.

In [12]:
if CREATE_MASTER_DATASET:
    master_dataset_path = processor.create_master_dataset(
        processed_dir=PROCESSED_PATH,
        output_path=MASTER_DATASET_PATH,
    )
else:
    master_dataset_path = MASTER_DATASET_PATH

print("Master dataset path:", master_dataset_path)

gc.collect()

2026-03-20 17:05:45 | INFO | wtfd.data.preprocessing | Creating master dataset from 95 event parquet files using PyArrow dataset
2026-03-20 17:05:46 | INFO | wtfd.data.preprocessing | Initialized ParquetWriter for /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/master_dataset.parquet
2026-03-20 17:06:33 | INFO | wtfd.data.preprocessing | Saved master dataset to /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/master_dataset.parquet | rows=5242948 | batches=96


Master dataset path: /mnt/c/grad_school/northeastern/courses/ie7275/project/ML_Project/data/processed/master_dataset.parquet


0

## Inspect master dataset metadata without loading the full file

In [13]:
row_count = processor.get_parquet_row_count(master_dataset_path)
file_size_mb = round(master_dataset_path.stat().st_size / (1024 ** 2), 3)

print("Master dataset rows:", row_count)
print("Master dataset size (MB):", file_size_mb)

Master dataset rows: 5242948
Master dataset size (MB): 1031.765


In [14]:
parquet_file = pq.ParquetFile(master_dataset_path)

print("Number of row groups:", parquet_file.num_row_groups)
print("Schema:")
print(parquet_file.schema)

Number of row groups: 96
Schema:
required group field_id=-1 schema {
  optional int64 field_id=-1 time_stamp (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional binary field_id=-1 farm_id (String);
  optional binary field_id=-1 asset_id (String);
  optional double field_id=-1 amb_temp;
  optional double field_id=-1 wind_speed;
  optional double field_id=-1 pitch_angle;
  optional double field_id=-1 active_power;
  optional double field_id=-1 gen_speed;
  optional double field_id=-1 gearbox_oil_temp;
  optional double field_id=-1 transformer_temp;
  optional double field_id=-1 nacelle_temp;
  optional double field_id=-1 hub_temp;
  optional double field_id=-1 yaw_error;
  optional double field_id=-1 vibration_raw;
  optional double field_id=-1 hydraulic_temp;
  optional double field_id=-1 generator_temp;
  optional double field_id=-1 temp_delta_gearbox;
  optional double field_id=-1 temp_trend_24h;
  optional

## Load a small sample for sanity checks

To reduce memory usage, only a limited subset of rows is loaded here.

In [26]:
sample_n = 5000

sample_df = pd.read_parquet(master_dataset_path).sample(
    n=min(sample_n, row_count),
    random_state=42,
)

print("Sample shape:", sample_df.shape)
sample_df.head()

Sample shape: (5000, 39)


,time_stamp,farm_id,asset_id,amb_temp,wind_speed,pitch_angle,active_power,gen_speed,gearbox_oil_temp,transformer_temp,...,generator_temp_volatility_24,active_power_delta_6,active_power_volatility_6,active_power_delta_24,active_power_volatility_24,yaw_error_delta_6,yaw_error_volatility_6,yaw_error_delta_24,yaw_error_volatility_24,target
1347700,2023-06-28 08:20:00,B,0,17.880,4.540,1.4900,0.038434,748.280000,59.5600,45.0100,...,2.448472,0.016260,0.013595,0.043716,0.052267,-5.430,6.232663,41.9650,9.766795,0
2060947,2022-11-05 15:50:00,C,53,28.245,8.494,-0.4252,0.029610,124.780658,56.0925,47.8310,...,4.078108,-0.770250,0.320976,-0.495790,0.206016,-4.642,3.733496,-1.5330,3.321633,0
3294553,2023-06-11 19:20:00,C,21,39.818,8.907,-1.5000,0.619540,1637.704364,54.0440,39.1085,...,1.572090,0.102420,0.048585,0.166440,0.082942,0.377,4.222419,5.0940,5.719884,0
873405,2022-09-18 06:50:00,A,11,23.000,5.700,-1.6000,95.966000,1308.400000,50.0000,64.0000,...,1.437086,17.878000,43.798736,85.318000,47.775788,-4.200,14.647560,-18.4000,9.148089,0
5223817,2023-02-16 14:20:00,C,34,27.509,5.647,-1.5000,0.193526,1101.129389,50.7160,29.4910,...,1.629182,-0.199094,0.023688,-0.146054,0.106944,3.237,3.211032,3.6463,3.380630,0


In [27]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 1347700 to 2614125
Data columns (total 39 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   time_stamp                      5000 non-null   datetime64[ns]
 1   farm_id                         5000 non-null   object        
 2   asset_id                        5000 non-null   object        
 3   amb_temp                        5000 non-null   float64       
 4   wind_speed                      5000 non-null   float64       
 5   pitch_angle                     5000 non-null   float64       
 6   active_power                    5000 non-null   float64       
 7   gen_speed                       5000 non-null   float64       
 8   gearbox_oil_temp                5000 non-null   float64       
 9   transformer_temp                5000 non-null   float64       
 10  nacelle_temp                    5000 non-null   float64       
 11  

## Basic sanity checks on the sample

In [28]:
print("Sample rows:", len(sample_df))
print("Sample columns:", sample_df.shape[1])
print("Number of farms in sample:", sample_df["farm_id"].nunique() if "farm_id" in sample_df.columns else "N/A")
print("Number of assets in sample:", sample_df["asset_id"].nunique() if "asset_id" in sample_df.columns else "N/A")
print("Sample time range:")
print(sample_df["time_stamp"].min(), "to", sample_df["time_stamp"].max())

Sample rows: 5000
Sample columns: 39
Number of farms in sample: 3
Number of assets in sample: 27
Sample time range:
2022-01-08 20:50:00 to 2024-02-06 12:10:00


In [29]:
missingness_df = (
    sample_df.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .reset_index()
    .rename(columns={"index": "column"})
)

missingness_df.head(20)

,column,missing_fraction
0,time_stamp,0.0
1,generator_temp_volatility_24,0.0
2,gearbox_oil_temp_delta_6,0.0
3,gearbox_oil_temp_volatility_6,0.0
4,gearbox_oil_temp_delta_24,0.0
5,gearbox_oil_temp_volatility_24,0.0
6,generator_temp_delta_6,0.0
7,generator_temp_volatility_6,0.0
8,generator_temp_delta_24,0.0
9,active_power_delta_6,0.0


In [30]:
if "target" in sample_df.columns:
    target_summary = pd.DataFrame(
        {
            "target_value": sample_df["target"].value_counts(dropna=False).index,
            "count": sample_df["target"].value_counts(dropna=False).values,
        }
    )
    target_summary["fraction"] = target_summary["count"] / target_summary["count"].sum()
    target_summary
else:
    print("No target column found.")

In [31]:
if {"farm_id", "asset_id"}.issubset(sample_df.columns):
    if "target" in sample_df.columns:
        farm_summary = (
            sample_df.groupby("farm_id")
            .agg(
                n_rows=("farm_id", "size"),
                n_assets=("asset_id", "nunique"),
                positive_labels=("target", "sum"),
            )
            .reset_index()
        )
    else:
        farm_summary = (
            sample_df.groupby("farm_id")
            .agg(
                n_rows=("farm_id", "size"),
                n_assets=("asset_id", "nunique"),
            )
            .reset_index()
        )
    farm_summary
else:
    print("farm_id and/or asset_id column not found.")

## Check expected feature presence in the sample

In [32]:
expected_features = set(processor.standard_features) | set(processor.config.get("derived_features", []))
available_features = set(sample_df.columns)

missing_expected_features = sorted(expected_features - available_features)
present_expected_features = sorted(expected_features & available_features)

print("Present expected features:", len(present_expected_features))
print("Missing expected features:", len(missing_expected_features))
missing_expected_features

Present expected features: 19
Missing expected features: 0


[]

In [33]:
present_expected_features[:50]

['active_power',
 'amb_temp',
 'gearbox_oil_temp',
 'gen_speed',
 'generator_temp',
 'hub_temp',
 'hydraulic_temp',
 'nacelle_temp',
 'pitch_angle',
 'power_efficiency',
 'rpm_volatility',
 'temp_delta_gearbox',
 'temp_divergence',
 'temp_trend_24h',
 'transformer_temp',
 'vibration_magnitude',
 'vibration_raw',
 'wind_speed',
 'yaw_error']

## Compile schema validation summary

In [34]:
# 1. YAML-declared mapped + named derived features
standard_features = set(processor.standard_features)
named_derived_features = set(processor.config.get("derived_features", []))

# 2. Code-generated temporal engineered features
lag_targets = {"gearbox_oil_temp", "generator_temp", "active_power", "yaw_error"}
windows = [6, 24]

temporal_features = {
    f"{col}_delta_{w}"
    for col in lag_targets
    for w in windows
} | {
    f"{col}_volatility_{w}"
    for col in lag_targets
    for w in windows
}

# 3. Metadata / non-feature columns typically expected in processed output
metadata_columns = {"time_stamp", "farm_id", "asset_id", "target"}

# 4. Available columns in the sampled dataframe
available_columns = set(sample_df.columns)

# 5. Missing columns by category
missing_standard = sorted(standard_features - available_columns)
missing_named_derived = sorted(named_derived_features - available_columns)
missing_temporal = sorted(temporal_features - available_columns)
missing_metadata = sorted(metadata_columns - available_columns)

# 6. Present columns by category
present_standard = sorted(standard_features & available_columns)
present_named_derived = sorted(named_derived_features & available_columns)
present_temporal = sorted(temporal_features & available_columns)
present_metadata = sorted(metadata_columns & available_columns)

# 7. Summary table
schema_summary = pd.DataFrame(
    [
        {
            "category": "standard_features",
            "expected": len(standard_features),
            "present": len(present_standard),
            "missing": len(missing_standard),
        },
        {
            "category": "named_derived_features",
            "expected": len(named_derived_features),
            "present": len(present_named_derived),
            "missing": len(missing_named_derived),
        },
        {
            "category": "temporal_features",
            "expected": len(temporal_features),
            "present": len(present_temporal),
            "missing": len(missing_temporal),
        },
        {
            "category": "metadata_columns",
            "expected": len(metadata_columns),
            "present": len(present_metadata),
            "missing": len(missing_metadata),
        },
        {
            "category": "total_expected_columns",
            "expected": len(
                standard_features
                | named_derived_features
                | temporal_features
                | metadata_columns
            ),
            "present": len(
                (
                    standard_features
                    | named_derived_features
                    | temporal_features
                    | metadata_columns
                )
                & available_columns
            ),
            "missing": len(
                (
                    standard_features
                    | named_derived_features
                    | temporal_features
                    | metadata_columns
                )
                - available_columns
            ),
        },
    ]
)

schema_summary

,category,expected,present,missing
0,standard_features,13,13,0
1,named_derived_features,6,6,0
2,temporal_features,16,16,0
3,metadata_columns,4,4,0
4,total_expected_columns,39,39,0


In [35]:
print("Missing standard features:", missing_standard)
print("Missing named derived features:", missing_named_derived)
print("Missing temporal features:", missing_temporal)
print("Missing metadata columns:", missing_metadata)

Missing standard features: []
Missing named derived features: []
Missing temporal features: []
Missing metadata columns: []


## Clean up large in-memory objects

In [36]:
del sample_df
gc.collect()
print("Released sample_df from memory.")

Released sample_df from memory.


## Preprocessing summary

This notebook completed the following steps:

1. Loaded the preprocessing configuration
2. Processed raw SCADA event files into standardized event-level Parquet files
3. Created a consolidated master dataset using the memory-aware PyArrow workflow
4. Inspected metadata and sampled rows without loading the full dataset into memory

The resulting `master_dataset.parquet` is the primary input for downstream modeling and evaluation workflows.